# Module 04 — Checkpointing

**Released:** v1.14.0 → v1.14.3 (April 2026)

Checkpointing persists the **entire live runtime** at event boundaries. With it you can:

- **Resume** a run that was killed mid-flight (OOM, Ctrl-C, transient LLM error).
- **Fork** a run — restart from step N with different inputs; new writes track back to the source via `parent_id`.
- **Audit** what the system looked like at any step: inputs, LLM calls so far, tool outputs, pending state.

It works on three kickoffs: `Flow.kickoff(from_checkpoint=…)`, `Crew.kickoff(from_checkpoint=…)`, and — new — `Agent.kickoff(from_checkpoint=…)`.

This notebook walks through:

1. How it actually works (event listener → serialize → provider)
2. The `CheckpointConfig` knobs that matter
3. The event vocabulary (incl. signals like `SIGINT`)
4. Providers: `JsonProvider` vs `SqliteProvider`
5. **Step-by-step: Flow kickoff with checkpointing + resume**
6. **Step-by-step: `Agent.kickoff()` with checkpointing + resume**
7. Forking with lineage
8. Extending: custom providers
9. CLI reference

## 1. How it works under the hood

```
┌────────────────┐    event fires    ┌──────────────────────┐
│  Flow / Crew / │──────────────────▶│  checkpoint_listener │
│  Agent runtime │                   └──────────┬───────────┘
└────────────────┘                              │
                                                ▼
             ┌──────────────────────────────────────────────────────┐
             │  for every active entity with a CheckpointConfig:    │
             │   if event type ∈ config.on_events (or "*"):         │
             │     data = RuntimeState.model_dump_json()            │
             │     id   = provider.checkpoint(                      │
             │              data, config.location,                  │
             │              parent_id=<prev id>, branch="main")     │
             │     if config.max_checkpoints: provider.prune(...)   │
             └──────────────────────────────────────────────────────┘
```

Key details:

- **`RuntimeState` is global.** Serializing it captures *every* active crew / flow / agent — not just the one that fired the event. That's why resuming a single flow can rehydrate its agents, pending tasks, and LLM context together.
- **Handlers are lazy.** The listener only hooks into the event bus the first time a `CheckpointConfig` is resolved. Zero overhead when nothing is checkpointed.
- **Writes are cheap.** `SqliteProvider` uses WAL; `JsonProvider` writes one file per checkpoint. Pruning is per-branch and happens inline after each write.
- **Lineage is on the row.** Every checkpoint stores `parent_id` and `branch`, so the full tree (main → fork → fork-of-fork) is queryable later via `crewai checkpoint info`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from crewai.state.checkpoint_config import CheckpointConfig
from crewai.state.provider.sqlite_provider import SqliteProvider

# The CLI's default lookup checks two paths:
#   ./.checkpoints      (JSON directory, JsonProvider convention)
#   ./.checkpoints.db   (SQLite file, SqliteProvider convention)
# Using one of these lets `crewai checkpoint list` / `resume` Just Work
# without needing to pass --location every time.
config = CheckpointConfig(
    location="./.checkpoints.db",
    provider=SqliteProvider(),
    on_events=["task_completed", "method_execution_finished"],
    max_checkpoints=10,
)
config

## 2. `CheckpointConfig` — the five fields

| Field | Default | Purpose |
|---|---|---|
| `location` | `./.checkpoints` | Directory (JsonProvider) or DB file path (SqliteProvider). Created on first write. |
| `provider` | required | `JsonProvider()` or `SqliteProvider()` — or any `BaseProvider` subclass. |
| `on_events` | `["task_completed"]` | Event types that trigger a write. `["*"]` captures everything. |
| `max_checkpoints` | `None` | Keep at most N per branch; older rows are pruned after each write. |
| `restore_from` | `None` | Checkpoint id, file path, or `"db#id"` location. Setting this turns `kickoff(from_checkpoint=config)` into a **resume** instead of a fresh run. |

## 3. The event vocabulary

`on_events` accepts any `CheckpointEventType` literal. There are ~100 of them — here are the groupings that matter:

| Group | Examples | When you'd pick these |
|---|---|---|
| **Crew lifecycle** | `crew_kickoff_started/completed/failed`, `task_started/completed/failed` | Coarse-grained checkpointing between task boundaries. |
| **Flow lifecycle** | `flow_started/finished/paused`, `method_execution_started/finished/failed/paused` | One checkpoint per Flow step. |
| **Agent lifecycle** | `agent_execution_started/completed/error`, `lite_agent_execution_*` | Track every Agent invocation; `lite_*` events fire for `Agent.kickoff()`. |
| **LLM** | `llm_call_started/completed/failed`, `llm_stream_chunk`, `llm_guardrail_*` | Debug LLM behaviour, retry from last successful call. |
| **Tools** | `tool_usage_started/finished`, `tool_execution_error` | Checkpoint before/after every tool call — great for long-running tool chains. |
| **Plan-and-Execute** | `step_observation_*`, `plan_refinement`, `plan_replan_triggered`, `goal_achieved_early` | Granular snapshots of the planner's internal loop (Module 02). |
| **Memory** | `memory_save_*`, `memory_query_*`, `memory_retrieval_*` | Audit trail of what was read / written to memory. |
| **A2A** | 25+ events (`a2a_delegation_*`, `a2a_streaming_*`, `a2a_server_task_*`, …) | Reproduce agent-to-agent conversations step-by-step. |
| **Signals** | `SIGINT`, `SIGTERM`, `SIGHUP`, `SIGTSTP`, `SIGCONT` | ⭐ **Checkpoint on Ctrl-C** — the listener intercepts the signal, snapshots, then re-raises. |
| **Env sentinels** | `cc_env`, `codex_env`, `cursor_env`, `default_env` | Fire when an agent session boots inside a particular coding-agent context. |

Common picks:

- `["task_completed"]` — safest default.
- `["task_completed", "task_failed"]` — post-mortem on failures.
- `["method_execution_finished"]` — Flow-step granularity.
- `["task_completed", "SIGINT", "SIGTERM"]` — normal completion + graceful-exit safety net.
- `["*"]` — everything. Big, but unbeatable for debugging.

## 4. Providers — `JsonProvider` vs `SqliteProvider`

| | `JsonProvider` (default) | `SqliteProvider` |
|---|---|---|
| Layout | One `.json` file per checkpoint in `location/` | Single `.db` file with WAL journaling |
| Best for | Dev, manual inspection (`cat checkpoint_xxx.json`) | Many checkpoints per run; bulk queries via SQL |
| Locking | Filesystem-level | `PRAGMA journal_mode=WAL` — concurrent readers OK |
| Lineage | `parent_id` field inside each JSON | `parent_id` column alongside `branch` |

Both share the same `BaseProvider` contract (6 methods: `checkpoint`, `acheckpoint`, `from_checkpoint`, `afrom_checkpoint`, `extract_id`, `prune`), so swapping providers is a one-line change.

## 5. Step-by-step — checkpoint a **Flow**

The `CheckpointFlow` (under `src/showcase/flows/checkpoint_flow.py`) builds a five-beat **micro-story** — one sentence per step, one `agent.kickoff()` per step, one checkpoint per step. Total run ~10–15 seconds, no tools, nothing external.

```
@start()  hook         → one-sentence hook          ✓ checkpoint
@listen   character    → introduce the character    ✓ checkpoint
@listen   conflict     → what goes wrong            ✓ checkpoint
@listen   twist        → the surprise turn          ✓ checkpoint
@listen   resolution   → how it ends                ✓ checkpoint
```

Default seed: *"a lighthouse keeper who's been alone for too long"* — override via `run_fresh(seed="…")`.

Wired with `SqliteProvider` at `./.checkpoints.db` (the CLI's default lookup path) and `on_events=["method_execution_finished"]`, so you get one checkpoint after every step — five total per full run.

Each step checks its state field (`hook`, `character`, `conflict`, `twist`, `resolution`) and short-circuits if already populated. Kill the run after the conflict beat, and a resume replays hook+character+conflict from state while only twist+resolution actually hit the LLM.

**Prereq:** `ANTHROPIC_API_KEY` (or `OPENAI_API_KEY` with `SHOWCASE_LLM=openai`).

### Step 1 — run it fresh

`run_fresh()` calls `flow.kickoff(from_checkpoint=CheckpointConfig(...))`. Because `restore_from` is not set, this is a *fresh run with checkpointing on* — not a resume.

In [ ]:
from showcase.flows.checkpoint_flow import run_fresh

flow = run_fresh()
print(flow.state.story)

### Step 2 — list what got written

After a full run, five checkpoints should be in the DB — one per step. If you killed the run mid-build you'll see fewer rows, newest first.

In [ ]:
# Bare `crewai checkpoint list` auto-detects ./.checkpoints.db
!crewai checkpoint list 2>&1 | tail -20

### Step 3 — resume from a specific checkpoint

Grab an id from the list above and pass it as `restore_from` **using the `"db#id"` form** — `SqliteProvider` splits on `#` to recover the DB path. On the resumed run:

- Steps whose state field is already populated short-circuit (no LLM call, no token cost).
- Steps whose state field is empty execute normally, using the already-populated beats as context.
- `resolution` also joins the five beats into `state.story`.

```python
from showcase.flows.checkpoint_flow import resume_from

# The "db#id" form — note the "./.checkpoints.db#" prefix.
resumed = resume_from("./.checkpoints.db#20260423T104512_ab12cd34")
print(resumed.state.story)
```

Or via the CLI (no code):

```bash
crewai checkpoint resume 20260423T104512_ab12cd34
```

**Try this in one sitting:** run `run_fresh()`, Ctrl-C after the conflict sentence prints, then `resume_from("./.checkpoints.db#<id>")` — you should see the first three beats reused verbatim while the twist and resolution are generated fresh.

→ See §7 for **forking** — same mechanism, plus input overrides and a new branch label.

## 6. Step-by-step — checkpoint an **`Agent.kickoff()`**

Checkpointing isn't Flow-only. Single agents can checkpoint too. This is useful when an agent is a long-running loop (tool calls, multi-turn reasoning) and you want restartability without wrapping it in a Flow.

### Step 1 — build an agent and kick it off with checkpointing

The same `CheckpointConfig` shape works — `Agent.kickoff()` accepts `from_checkpoint=...` just like `Flow.kickoff()`.

In [ ]:
from crewai import Agent
from crewai.state.checkpoint_config import CheckpointConfig
from crewai.state.provider.sqlite_provider import SqliteProvider
from showcase.shared import get_llm

# Reuse the CLI default path — flow + agent checkpoints land in the same DB.
# They're distinguishable via the `entities` column in `crewai checkpoint list`,
# and `crewai checkpoint resume <id>` picks the right entity type to rehydrate.
CHECKPOINT_DB = "./.checkpoints.db"

agent = Agent(
    role="Architecture Analyst",
    goal="Explain a piece of architecture clearly",
    backstory="Senior engineer who writes crisp technical explainers.",
    llm=get_llm(),
)

fresh = CheckpointConfig(
    location=CHECKPOINT_DB,
    provider=SqliteProvider(),
    # lite_agent_* are the events Agent.kickoff fires
    on_events=["lite_agent_execution_completed"],
    max_checkpoints=5,
)

# Inside Jupyter `agent.kickoff()` returns a coroutine — top-level await unwraps it.
first = await agent.kickoff(
    "In 80 words, explain why WAL-mode SQLite is a good fit for a per-process "
    "checkpoint store. Focus on concurrency and crash safety.",
    from_checkpoint=fresh,
)
print(first.raw)

### Step 2 — resume with a new input (context threading)

Pass a **different** `messages=` argument to a second `agent.kickoff()` with `restore_from` set. One subtlety worth flagging up front: the snapshot captures the agent's **config** (role, goal, tools, usage counters, memory binding, checkpoint config) — **not** the prior kickoff's conversation scratchpad. To give the resumed call that context, thread the prior output into the new prompt yourself.

**What the resume still buys you:**
- Identical agent config + tool state (usage counters carry over, so rate-limited tools don't reset).
- Lineage tracking via `parent_id` — every new checkpoint on the resumed call points at the snapshot you resumed from.
- Same DB, same pruning rules, same branch label.

**What you handle manually:**
- Prior conversation context — thread `first.raw` (or whatever subset you need) into the new prompt.

For multi-turn agents that need persistent memory across many kickoffs, give the agent a `Memory` (see [Module 03](./03_unified_memory.ipynb)); the memory recall handles cross-turn context so you don't have to thread outputs by hand.

In [ ]:
import sqlite3

# Grab the latest agent checkpoint id — in a real session you'd usually pick
# one from `crewai checkpoint list` output.
with sqlite3.connect(CHECKPOINT_DB) as db:
    ck_id = db.execute(
        "SELECT id FROM checkpoints "
        "WHERE json(data) LIKE '%Architecture Analyst%' "
        "ORDER BY rowid DESC LIMIT 1"
    ).fetchone()[0]
print("resuming from:", ck_id)

resume = CheckpointConfig(
    location=CHECKPOINT_DB,
    provider=SqliteProvider(),
    on_events=["lite_agent_execution_completed"],
    max_checkpoints=5,
    restore_from=f"{CHECKPOINT_DB}#{ck_id}",
)

# Thread the first output into the follow-up so the agent has grounded context.
second = await agent.kickoff(
    f"You previously wrote:\n\n{first.raw}\n\n"
    "Now name TWO specific failure modes to watch for in that design. "
    "One sentence each.",
    from_checkpoint=resume,
)
print("\n---\n")
print(second.raw)

In [ ]:
# Flow + agent checkpoints live in the same DB. Newest rows first — the
# `entities` column tells you whether a row came from CheckpointFlow or
# the standalone Agent.kickoff above.
!crewai checkpoint list 2>&1 | tail -10

## 7. Forking with lineage

Forking = resume from a checkpoint, **override state**, and continue on a new branch. The provider writes each new checkpoint with `parent_id = <source>` and `branch = <label>`. Pruning is per-branch, so experimental forks never clobber `main`.

The flow module ships a `fork_from(checkpoint_path, branch, **state_overrides)` helper. It uses CrewAI's `Flow.fork(cfg, branch=...)` under the hood and applies your overrides to `flow.state` before kickoff:

```python
from showcase.flows.checkpoint_flow import fork_from

# Resume from the "after conflict" checkpoint, but force a completely
# different conflict. Twist + resolution will regenerate using the new value.
forked = fork_from(
    "./.checkpoints.db#20260423T104512_ab12cd34",
    branch="alt-conflict",
    conflict="A stranger is knocking at the lighthouse door.",
)
print(forked.state.story)
```

What happens inside:

1. `CheckpointFlow.fork(cfg, branch="alt-conflict")` — rehydrates state from the checkpoint and tags the runtime with the new branch label.
2. `flow.checkpoint = cfg` (minus `restore_from`) — so subsequent checkpoints keep writing to the same DB on the new branch.
3. `setattr(flow.state, "conflict", "...")` — overrides any state field before kickoff.
4. `flow.kickoff()` — the already-completed steps (hook, character, conflict) short-circuit on their state flags; twist + resolution run fresh, informed by the overridden conflict.

Verify the fork landed on its own branch:

```python
import sqlite3
with sqlite3.connect(".checkpoints.db") as db:
    for row in db.execute(
        "SELECT id, branch, parent_id FROM checkpoints ORDER BY rowid DESC LIMIT 4"
    ):
        print(row)
```

You should see the new rows carrying `branch="alt-conflict"` with `parent_id` pointing at your source checkpoint. `crewai checkpoint info <id>` prints the same lineage from the CLI side.

## 8. Extending — your own provider

Subclass `crewai.state.provider.core.BaseProvider` and implement six methods:

```python
class PostgresProvider(BaseProvider):
    provider_type = "postgres"

    def checkpoint(self, data, location, *, parent_id=None, branch="main") -> str: ...
    async def acheckpoint(self, data, location, *, parent_id=None, branch="main") -> str: ...
    def from_checkpoint(self, location) -> str: ...
    async def afrom_checkpoint(self, location) -> str: ...
    def extract_id(self, location) -> str: ...
    def prune(self, location, max_keep, *, branch="main") -> None: ...
```

Then pass an instance: `CheckpointConfig(provider=PostgresProvider(dsn=...), ...)`. Everything else — the listener, event routing, lineage tracking — works unchanged.

## 9. CLI reference

`--location` is a **group-level** option — it must go *before* the subcommand. The default is `./.checkpoints` (as a JSON dir); if that path doesn't exist but `./.checkpoints.db` does, the CLI auto-picks it.

```
# Bare commands — uses default lookup (./.checkpoints/ or ./.checkpoints.db)
crewai checkpoint list
crewai checkpoint info
crewai checkpoint resume           # most recent
crewai checkpoint resume <ID>

# Custom DB location — pass before the subcommand:
crewai checkpoint --location path/to/my.db list
crewai checkpoint --location path/to/my.db resume <ID>
crewai checkpoint --location path/to/my.db diff <ID1> <ID2>
crewai checkpoint --location path/to/my.db prune
```

`info` shows the serialized `RuntimeState` snapshot — it's the closest thing to a debugger you have when a run behaves unexpectedly.